In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('vader_lexicon')


gsheet_url = 'https://docs.google.com/spreadsheets/d/'
sheet_id = '1xHgmmhVE4IA7F60qnNkuMQ6OxVpR7AEx'

data = pd.read_excel(f'{gsheet_url}{sheet_id}/export?format=xlsx')

def column_cleaner(selected_column):
    data[selected_column] = data[selected_column].str.lower()
    data[selected_column] = data[selected_column].str.strip()
    data[selected_column].drop_duplicates(keep='first', inplace=True)
    return data

data = column_cleaner('Key Words')

lemmatizer = WordNetLemmatizer()
english_stop_words = set(stopwords.words('english'))
filipino_stop_words = set(stopwords.words('filipino'))
chinese_stop_words = set(stopwords.words('chinese'))
stop_words = english_stop_words | filipino_stop_words | chinese_stop_words

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

data['Key Words'] = data['Key Words'].apply(preprocess_text)

data['Comment Length'] = data['Key Words'].apply(lambda x: len(x.split()))

X_train, X_val, y_train, y_val = train_test_split(data[['Key Words', 'Comment Length']], data['Category'], test_size=0.2, random_state=42)

text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000)),
    ('clf', MultinomialNB())
])

param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_df': [0.7, 0.8, 0.9],
    'clf__alpha': [0.1, 0.5, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cross_val_scores = cross_val_score(text_clf, X_train['Key Words'], y_train, cv=cv, scoring='accuracy')

grid_search = GridSearchCV(text_clf, param_grid, cv=cv, n_jobs=-1)
grid_search.fit(X_train['Key Words'], y_train)

print("Best hyperparameters: ", grid_search.best_params_)

y_pred = grid_search.predict(X_val['Key Words'])

accuracy = accuracy_score(y_val, y_pred)
report = classification_report(y_val, y_pred)

print(f'Accuracy: {accuracy}')
print(report)

sid = SentimentIntensityAnalyzer()

def analyze_sentiment(comment):
    sentiment_scores = sid.polarity_scores(comment)
    if sentiment_scores['compound'] >= 0.05:
        return 'positive'
    elif sentiment_scores['compound'] <= -0.05:
        return 'negative'
    else:
        return 'neutral'
    
data['Sentiment'] = data['Key Words'].apply(analyze_sentiment)

target_path = 'TestComments.xlsx'
target_comment = pd.read_excel(target_path)
target_comments = target_comment['Comments'].tolist()

sentiments = [analyze_sentiment(comment) for comment in target_comments]
X_target = pd.DataFrame({'Reviews': target_comments, 'Sentiment': sentiments})
predicted_categories = grid_search.predict(X_target['Reviews'])

df = pd.DataFrame({'Reviews': target_comments, 'Category': predicted_categories, 'Sentiment': sentiments})

df

[nltk_data] Downloading package punkt to C:\Users\Jerome
[nltk_data]     Pintucan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Jerome
[nltk_data]     Pintucan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Jerome
[nltk_data]     Pintucan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package vader_lexicon to C:\Users\Jerome
[nltk_data]     Pintucan\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
c:\Users\Jerome Pintucan\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\Jerome Pintucan\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\mod

Best hyperparameters:  {'clf__alpha': 0.1, 'tfidf__max_df': 0.7, 'tfidf__ngram_range': (1, 2)}
Accuracy: 0.8666666666666667
                        precision    recall  f1-score   support

Food Quality and Taste       0.91      0.93      0.92        85
        Order Accuracy       0.75      0.75      0.75        20
               Portion       1.00      0.78      0.88         9
               Pricing       0.50      0.50      0.50         6

              accuracy                           0.87       120
             macro avg       0.79      0.74      0.76       120
          weighted avg       0.87      0.87      0.87       120



,Reviews,Category,Sentiment
0,food amazing delicious,Food Quality and Taste,positive
1,order mess forgot dessert wrong,Order Accuracy,negative
2,requested tomatoes salad extra tomatoes but I ...,Order Accuracy,neutral
3,prices steep taste worth penny,Food Quality and Taste,positive
4,portion sizes generous finish meal,Portion,positive
5,sushi fresh flavorful culinary delight,Food Quality and Taste,positive
6,remind server multiple times missing appetizer,Order Accuracy,negative
7,burger dry overcooked expected,Food Quality and Taste,neutral
8,small portion prices high,Pricing,neutral
9,extra sauce pasta barely drizzled,Food Quality and Taste,neutral
